# Activity: Singular Value Decomposition (SVD) of the Stoichiomatric Matrix for Genome-Scale Metabolic Models
In this activity, we will perform Singular Value Decomposition (SVD) on the stoichiometric matrix of a genome-scale metabolic model (GEM) and analyze the results to gain insights into the metabolic capabilities of the organism. Thus, we continue our analysis from the previous module of the covariance matrix computed from the stiochiometric matrix.

> __Learning Objectives:__
> 
> By the end of this activity, you will be able to:
>
> Three learning objectives here.

Let's get started!
___

## Background: What is a stoichiometric matrix?
Suppose we have a set of chemical (or biochemical) reactions $\mathcal{R}$ involving the chemical species (metabolite) set $\mathcal{M}$. Then, the stoichiometric matrix is a $\mathbf{S}\in\mathbb{R}^{|\mathcal{M}|\times|\mathcal{R}|}$ matrix that holds the stoichiometric coefficients $\sigma_{ij}\in\mathbf{S}$ such that:
* $\sigma_{ij}>0$: Chemical species (metabolite) $i$ is _produced_ by reaction $j$. Species $i$ is a product of reaction $j$.
* $\sigma_{ij} = 0$: Chemical species (metabolite) $i$ is not connected with reaction $j$
* $\sigma_{ij}<0$: Chemical species (metabolite) $i$ is _consumed_ by reaction $j$. Species $i$ is a reactant of reaction $j$.

Thus, the stoichiometric matrix $\mathbf{S}$ encodes the complete connectivity information of the chemical reaction system for, in this case, a biochemical reaction network. Thus, it is the digital representation of the reaction network inside of a cell.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Data
We developed a simple software development kit (SDK) against [the BiGG Models application programming interface at the University of California, San Diego](http://bigg.ucsd.edu/). The [BiGG Models database](http://bigg.ucsd.edu/) integrates published genome-scale metabolic networks into a single database with standardized nomenclature and structure. 

> __What are we doing here?__
> 
> We are going to download a stoichiometric matrix from [the BiGG models database](http://bigg.ucsd.edu/) using [the BiGG models API](http://bigg.ucsd.edu/data_access) and then compute its eigendecomposition. 
> * [The BiGG models API](http://bigg.ucsd.edu/data_access) allows users to programmatically access genome-scale stoichiometric model reconstructions using a simple web API. There are `108` models of intracellular biochemistry occurring in various organisms (including humans) in the database (so far); [see here for a list of models](http://bigg.ucsd.edu/models).
> * Here, we'll first explore the [model of Platelet metabolism developed by Palsson and coworkers](https://pubmed.ncbi.nlm.nih.gov/24473230/), which is a curated, functionally tested, and experimentally validated biochemical reaction network of Human platelet metabolism. This model has 738 metabolites and 1008 reactions. 
> 
>  We call the model download endpoint of [the BiGG models API](http://bigg.ucsd.edu/data_access) and then save the model file to disk (so we don't hit the API unless we have to). 

This call returns model information organized as [a Julia dictionary](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) in the `model::Dict{String, Any}` variable. If a model file is saved, we use the cached file instead of making an API call.

In [2]:
model = let

    # build download endpoint -
    baseurl = "http://bigg.ucsd.edu"; # base url to download model
    modelid = "iAT_PLT_636"; # model id to download (change as needed)
    path_to_saved_model_file = joinpath(_PATH_TO_DATA, "saved-model-$(modelid).jld2");

    # check: do we have a model file saved?
    model = nothing;
    if (isfile(path_to_saved_model_file) == false)
        
        endpoint = MyBiggModelsDownloadModelEndpointModel();
        endpoint.bigg_id = modelid;
        url = build(baseurl, endpoint)
        model = MyBiggModelsDownloadModelEndpointModel(url);

        # Before we move on, save this model for later (so we don't keep hitting the API)
        save(path_to_saved_model_file, Dict("model" => model));
    else
        model = load(path_to_saved_model_file)["model"];
    end
    model; # return the model (either saved, or downloaded)
end

JSON.Object{String, Any} with 6 entries:
  "metabolites"  => Any[Object{String, Any}("id"=>"pa_hs_18_2_20_4_c", "name"=>…
  "reactions"    => Any[Object{String, Any}("id"=>"PI4P5K_18_0_20_4", "name"=>"…
  "genes"        => Any[Object{String, Any}("id"=>"8611", "name"=>"PLPP1", "not…
  "id"           => "iAT_PLT_636"
  "compartments" => Object{String, Any}("c"=>"cytosol", "e"=>"extracellular spa…
  "version"      => "1"

__Metabolite records__: Each metabolite (chemical compound) in the network has an associated metabolite record with several fields. Let's take a look at the metabolite at index `1`. The key field for today in the metabolite record is the `id` field, an abbreviation or symbol associated with this metabolite.

In [3]:
model["metabolites"][1] # example metabolite record

JSON.Object{String, Any} with 5 entries:
  "id"          => "pa_hs_18_2_20_4_c"
  "name"        => "Pa hs 18 2 20 4[c]"
  "compartment" => "c"
  "notes"       => Object{String, Any}("original_bigg_ids"=>Any["pa_hs_18_2_20_…
  "annotation"  => Object{String, Any}("bigg.metabolite"=>Any["pa_hs_18_2_20_4"…

__Reaction records__: Similarly, each reaction in the network has a reaction record with several fields. Let's look at the reaction record at index `25`. The key field for the reaction record is the `metabolites` field, which lists the stoichiometric coefficients associated with this particular reaction.

In [4]:
model["reactions"][25] # example reaction record

JSON.Object{String, Any} with 9 entries:
  "id"                 => "PI4P5K_18_1_18_2"
  "name"               => "PI4P5K 18 1 18 2"
  "metabolites"        => Object{String, Any}("adp_c"=>1.0, "atp_c"=>-1.0, "h_c…
  "lower_bound"        => 0.0
  "upper_bound"        => 1000.0
  "gene_reaction_rule" => "200576 or 23396 or 8394 or 8395 or 8396 or 5305 or 7…
  "subsystem"          => "Expanded Glycerophospholipid metabolism"
  "notes"              => Object{String, Any}("original_bigg_ids"=>Any["PI4P5K_…
  "annotation"         => Object{String, Any}("bigg.reaction"=>Any["PI4P5K_18_1…

For each reaction record, we can see the chemical species (metabolites) involved in the reaction and their associated stoichiometric coefficients. Negative coefficients indicate reactants (consumed), while positive coefficients indicate products (produced).

In [5]:
model["reactions"][25]["metabolites"]

JSON.Object{String, Any} with 5 entries:
  "adp_c"                   => 1.0
  "atp_c"                   => -1.0
  "h_c"                     => 1.0
  "pail345p_hs_18_1_18_2_c" => 1.0
  "pail45p_hs_18_1_18_2_c"  => -1.0

__Stoichiometric matrix__: Next, let's build a stoichiometric matrix $\mathbf{S}$ using the metabolite and reaction records. We'll do this using two for loops. 

> __Strategy__: In the outer loop, we iterate over the system's metabolites (chemical species) and select the `id` from the metabolites record for each metabolite. In the inner loop, we iterate over each reaction. For each reaction record, we ask if this reaction has an entry for the current metabolite `id` value; if it does, we grab the stoichiometric coefficient $\sigma_{ij}$ corresponding to this metabolite and reaction.

We'll save the stoichiometric matrix in the `S::Matrix{Float64}` variable.

In [6]:
S = let

    # get some data from the model -
    m = model["metabolites"]; # get list of metabolites
    r = model["reactions"]; # get list of reactions
    number_of_rows = length(m); # how many metabolites do we have? (rows)
    number_of_cols = length(r); # how many reactions do we have? (cols)
    S = zeros(number_of_rows,number_of_cols); # initialize an empty stoichiometric matrix

    # let's build a stm -
    for i ∈ eachindex(m)
        metabolite = m[i]["id"]; # we are checking if this metabolite is in the reaction record
        for j ∈ eachindex(r)
            reaction = r[j];
            if (haskey(reaction["metabolites"], metabolite) == true)
                S[i,j] = reaction["metabolites"][metabolite];
            end
        end
    end
    S; 
end;

___

## Task 1: Perform Singular Value Decomposition (SVD) on the Stoichiometric Matrix
In this task, we will perform Singular Value Decomposition (SVD) on the stoichiometric matrix $\mathbf{S}$ to analyze its properties and gain insights into the metabolic capabilities of our example metabolic model. We'll use [the built-in `svd(...)` function from Julia's `LinearAlgebra` standard library](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.svd) to compute the SVD of the stoichiometric matrix $\mathbf{S}$, where we'll use QR iteration as the algorithm for computing the SVD.

The SVD decomposes the matrix into three components: $\mathbf{U}$, $\mathbf{\Sigma}$, and $\mathbf{V}^{\top}$, where:
$$
\begin{align*}
\mathbf{S} &= \mathbf{U} \mathbf{\Sigma} \mathbf{V}^{\top} \\
\end{align*}
$$
where $\mathbf{U}$ is the left singular vectors matrix, $\mathbf{\Sigma}$ is the diagonal matrix of singular values, and $\mathbf{V}^{\top}$ is the transpose of the right singular vectors matrix. We'll store the results in the variables `U::Array{Float64,2}`, `Σ::Array{Float64,1}`, and `V::Array{Float64,2}` arrays.

In [7]:
U, Σ, V = svd(S, alg = LinearAlgebra.QRIteration()); # perform SVD on the stoichiometric matrix S. The full = true argument ensures that we get the full-sized U and V matrices.

What is contained in the `U`, `Σ`, and `V` matrices? How can we interpret these results in the context of the metabolic model? Let's start by looking at the dimensions of these matrices.

The `U::Array{Float64,2}` matrix has dimensions $|\mathcal{M}| \times |\mathcal{M}|$, where $|\mathcal{M}|$ is the number of metabolites (chemical compounds) in the model. Each column of `U::Array{Float64,2}` represents a left singular vector, which can be interpreted as a basis vector in the __metabolite space__. 

In [8]:
U

738×738 Matrix{Float64}:
  0.000687032   0.000104269  -0.00424429   …   1.25743e-17   1.90095e-39
 -0.254599     -0.0463165     0.0792987       -9.49291e-18   4.27311e-19
  0.0260666     0.00419208   -0.147488         1.23539e-17   6.27017e-19
 -7.92587e-5    2.49503e-6    0.00245063       7.45322e-19  -2.06108e-18
  0.00081002    0.000166228   0.00371522      -0.000453529  -1.40324e-18
  9.6321e-5     2.0074e-5     0.000618674  …  -1.87406e-17   2.25565e-18
 -0.17317      -0.0317587    -0.219571        -0.000453529  -5.49912e-20
 -0.0009244    -0.000153079   0.0038183       -3.77531e-17   4.03565e-18
 -7.77694e-5    2.79257e-6    0.00244449       1.48099e-17   1.84656e-19
  0.766963      0.111824     -0.353321         6.83176e-18  -2.01072e-19
  ⋮                                        ⋱                
  0.00399947    0.000703393  -0.023851         1.08054e-17  -1.69939e-18
 -1.43399e-6   -2.3594e-7    -0.000109645  …   6.51393e-17  -3.54544e-18
 -1.1307e-6    -2.72315e-7    3.16673e

What about the $\mathbf{V}$ matrix? The `V::Array{Float64,2}` matrix has dimensions $|\mathcal{R}| \times |\mathcal{M}|$, where $|\mathcal{R}|$ is the number of reactions in the model, and $|\mathcal{M}|$ is the number of metabolites. Each column of `V::Array{Float64,2}` represents a right singular vector, which can be interpreted as a basis vector in the __reaction space__.

Why isn't the `V::Array{Float64,2}` matrix $|\mathcal{R}| \times |\mathcal{R}|$ in size? This is the difference between the _full_ and _thin_ singular value decomposition. Here, we are using the thin SVD, which is more computationally efficient for large matrices like stoichiometric matrices in genome-scale metabolic models.

In [9]:
V

1008×738 adjoint(::Matrix{Float64}) with eltype Float64:
  0.0212228    0.00335473   -0.0621781   …   0.000120783   0.00921308
  0.0212009    0.00335089   -0.0619823      -0.00385801   -0.0729065
  0.0211946    0.00334998   -0.061743       -0.00385801   -0.0729065
  0.0244105    0.00400717   -0.0231099      -0.00444997    0.0894898
  0.0211822    0.00334773   -0.0617898      -0.00830799    0.0165834
  0.0244082    0.00400665   -0.0231916   …   0.0157374     0.0159184
  0.0187972    0.00291434   -0.0214122       0.00742945    0.0325018
 -0.018546    -0.00286991    0.020063        0.00742945    0.0325018
  0.0244325    0.00401056   -0.0232684      -0.0314412    -0.0277163
 -0.021199    -0.00335002    0.0620275       0.0374346     0.0943304
  ⋮                                      ⋱                
 -1.03207e-5  -1.78943e-6    5.03134e-5     -0.00617139   -0.00443884
  4.71266e-8   9.22567e-9   -6.13362e-7  …  -0.00503099    0.0190371
 -4.71266e-8  -9.22567e-9    6.13362e-7      0.0050309

How about the $\mathbf{\Sigma}$ matrix? The singular values returned by [the `svd(...)` functio](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.svd) are in a vector form (sorted from largest to smallest), in particular a $|\mathcal{M}|$-dimensional vector. We can work with this vector directly (as we do below), but often it can be useful to also have the singular values in matrix form.

In [10]:
Σ

738-element Vector{Float64}:
 42.2251127829501
 39.86664162745628
 19.43890586943346
 16.939678286740037
 16.665189786005918
 14.731745080044913
 13.058448297640798
 10.854151298687569
  9.347410133775515
  9.207570468189653
  ⋮
  1.948265613677479e-15
  1.872594860364972e-15
  1.813254709316857e-15
  1.6756894354566814e-15
  1.6078163339520589e-15
  1.4926003891701124e-15
  1.3242892159162612e-15
  6.641573947412779e-16
  2.973884242771285e-31

To convert this vector into a diagonal matrix, we can use [the `diagm(...)` function from Julia's standard library](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.diagm). The diagonal entries of the $\mathbf{\Sigma}$ matrix represent the singular values, which provide insights into the rank and condition number of the stoichiometric matrix $\mathbf{S}$. 

> __Rank and Condition Number:__
>
> * __Rank__: The number of non-zero singular values indicates the rank of the matrix, $r(\mathbf{S})\leq \min(|\mathcal{M}|, |\mathcal{R}|)$. A matrix with full rank (i.e., rank equal to the smaller of the number of rows or columns) indicates that the reactions in the metabolic model are linearly independent, meaning that each reaction contributes unique information to the system. On the other hand, a rank-deficient matrix (i.e., rank less than the smaller of the number of rows or columns) indicates that some reactions can be expressed as linear combinations of other reactions. 
> * __Condition Number__: The condition number of the stoichiometric matrix $\mathbf{S}$ can be computed as the ratio of the largest singular value to the smallest non-zero singular value. A high condition number indicates that the matrix is ill-conditioned, which can lead to numerical instability in computations involving the matrix, such as solving linear systems or performing optimization.

Let's check out these connections, by first computing the rank of the stoichiometric matrix $\mathbf{S}$ using [the `rank(...)` function from Julia's `LinearAlgebra` standard library](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.rank) and then counting the number of non-zero singular values in the `Σ::Array{Float64,1}` array.

In [11]:
let

    # initialize -
    ϵ = 1e-10; # threshold for considering singular values as non-zero
    r = rank(S); # compute the rank of the stoichiometric matrix S
    (M,R) = size(S); # get dimensions of S

    # how many non-zero singular values do we have?
    num_nonzero_singular_values = sum(σ -> σ > ϵ, Σ); # wow! that is some fancy functional programming there!

    # print results -
    println("Rank of the stoichiometric matrix S: $r");
    println("Number of non-zero singular values: $num_nonzero_singular_values");

    # Is the stoichiometric matrix S rank-deficient or full rank?
    if r < min(M,R)
        println("The stoichiometric matrix S is rank-deficient.");
    else
        println("The stoichiometric matrix S has full rank.");
    end
end

Rank of the stoichiometric matrix S: 719
Number of non-zero singular values: 719
The stoichiometric matrix S is rank-deficient.


What does it mean that the stoichiometric matrix $\mathbf{S}$ is rank-deficient? This suggest that some rows or columns of the matrix can be expressed as linear combinations of other rows or columns. In the context of metabolic networks, this implies that some reactions are not independent and can be derived from others. 

Next, if this matrix is rank deficient, we expect the condition number to be large. Let's compute the condition number of the stoichiometric matrix $\mathbf{S}$ using the singular values in the `Σ::Array{Float64,1}` array.

In [ ]:
let

    # initialize
    ϵ = 1e-10; # threshold for considering singular values as non-zero
    σ₁ = Σ[1]; # largest singular value
    σ₂ = findall(σ -> σ > ϵ, Σ) |> i-> Σ[i[end]]; # findall indices of non-zero singular values, get the last index, and use that to get the smallest non-zero singular value
   
    # compute condition number
    condition_number = σ₁ / σ₂; # condition number: max singular value / min non-zero singular value

    # print results
    println("Condition number of the stoichiometric matrix S: $condition_number");
end

Condition number of the stoichiometric matrix S: 1307.2069939575724


## Task 2: What insights can we gain from the SVD results?
In this task, we will analyze the singular values and singular vectors obtained from the SVD of the stoichiometric matrix $\mathbf{S}$ to gain insights into the metabolic capabilities of the organism represented by the model.

### Flux modes
Let's start thinking about the expansion of the stoichiometric matrix $\mathbf{S}$ in terms of its SVD components (flux modes):
$$
\begin{align*}
\mathbf{S} &= \sum_{i=1}^{r} \sigma_i \mathbf{u}_i \mathbf{v}_i^{\top} \\
\end{align*}
$$
where $r$ is the rank of the matrix, $\sigma_i$ are the singular values, $\mathbf{u}_i$ are the left singular vectors, and $\mathbf{v}_i$ are the right singular vectors. Then a clean interpretation is given by:

* $\mathbf{v}_k$ = a **reaction (flux) pattern** (a particular weighted combination of reactions),
* $\mathbf{u}_k$ = the resulting **metabolite net-production pattern** (a weighted combination of metabolites),
* $\sigma_k$ = the **gain/strength** that links them.

In fact, the singular triplets satisfy
$$
\mathbf{S}\mathbf{v}_k=\sigma_k,\mathbf{u}_k,
\qquad
\mathbf{S}^\top\mathbf{u}_k=\sigma_k,\mathbf{v}_k.
$$
Thus, if you push a unit of flux mode $\mathbf{v}_k$ into the network, the network produces the metabolite-change mode $\mathbf{u}_k$ scaled by $\sigma_k$. Let's check this out for the first singular triplet (the one with the largest singular value).

In [ ]:
let
end

Next, let's interpret each of the SVD matrices in biochemical terms.

### $\mathbf{U}$: orthonormal metabolite modes

The columns of $\mathbf{U}$ form an orthonormal basis for metabolite space, partitioned into two complementary subspaces:

* The first $r$ columns span the **column space** of $\mathbf{S}$—all achievable net metabolite production/consumption patterns
* The remaining $m-r$ columns span the **left nullspace**—the **conservation relations**

#### Conservation Relations

A **conservation relation** is a weighted sum of metabolites with a constant total. If $\mathbf{w}$ satisfies $\mathbf{S}^\top\mathbf{w}=\mathbf{0}$, then $\mathbf{w}^\top\mathbf{x}(t)$ is invariant (never changes with time). This represents pools like:
* Total adenylate pool: ATP + ADP + AMP
* Total redox cofactors: NAD$^+$ + NADH
* Phosphate pools and other conserved moieties

By SVD, the columns of $\mathbf{U}$ associated with zero singular values provide an **orthonormal basis** for these conservation relations.

> **For the mathematical derivation** of why the columns of $\mathbf{U}$ with zero singular values exactly span the left nullspace $N(\mathbf{S}^\top)$, see the companion notebook **CHEME-151-M3-Derivation-U-STM-Watch-Studio**. There we prove that $\mathbf{S}^\top\mathbf{U}_0 = \mathbf{0}$ through the SVD structure and show why this works.

#### Extracting Conservation Relations

In practice, we:
1. Identify columns of $\mathbf{U}$ with singular values $\sigma_k \leq \tau$, where $\tau = \epsilon \max(m,n)\sigma_1$ (here $\epsilon$ is machine precision, typically $\sim 10^{-15}$)
2. These columns form an orthonormal basis for all conservation relations
3. Any linear combination $\mathbf{w} = \mathbf{U}_0 \mathbf{a}$ is a valid conservation relation

Let's extract and examine the conservation relations from our metabolic model:

In [ ]:
let

    # Tolerance for identifying zero singular values
    (m, n) = size(S);
    ϵ = eps(Float64);  # machine precision
    τ = ϵ * max(m, n) * Σ[1];  # tolerance threshold
    
    # Find indices of singular values below threshold (zero singular values)
    zero_sv_indices = findall(σ -> σ <= τ, Σ);
    
    # Extract conservation relation vectors (columns of U with zero singular values)
    if !isempty(zero_sv_indices)
        U_0 = U[:, zero_sv_indices];  # basis for left nullspace (conservation relations)
        num_conservation = length(zero_sv_indices);
        
        println("Number of independent conservation relations: $num_conservation");
        println("Dimension of conservation relation basis: $(size(U_0))");
        
        # Display the first conservation relation as an example
        println("\nFirst conservation relation (weights for each metabolite):");
        println("Shape: $(size(U_0[:,1]))");
        
        # Count non-zero entries to understand structure
        w_example = U_0[:, 1];
        nonzero_entries = sum(w -> abs(w) > 1e-6, w_example);
        println("Number of metabolites involved: $nonzero_entries");
        
        # Verify it's actually a conservation relation: S' * w should be ≈ 0
        verification = norm(S' * w_example);
        println("||S' * w|| (should be ≈ 0): $verification");
    else
        println("No conservation relations detected (full rank stoichiometric matrix)");
    end
end

___

### $\mathbf{V}$: orthonormal reaction/flux modes
Similarly, the columns of the matrix $\mathbf{V}$ are an orthonormal basis for reaction space:

* The first $r$ columns ${\mathbf{v}_1,\dots,\mathbf{v}_r}$ span the **row space** of $\mathbf{S}$ (equivalently, the orthogonal complement of the right nullspace).
* The remaining $n-r$ columns span the **right nullspace** of $\mathbf{S}$.

That right nullspace is the classic steady-state story in constraint-based modeling:
$$
\mathbf{S}\mathbf{v}=0
$$
meaning the flux vector $\mathbf{v}$ produces **no net accumulation** of metabolites (internal steady-state mass balance). The right-nullspace basis vectors correspond to:

* **feasible steady-state flux degrees of freedom**, and/or
* **internal cycles** (circulations) depending on reversibility constraints and exchange reactions.

Again: SVD gives an orthonormal basis for that nullspace; other methods (EFMs, elementary flux vectors, integer bases) aim for interpretability/sparsity, but they live in the same subspace.

### $\mathbf{\Sigma}$: strengths + rank/conditioning

$\mathbf{\Sigma}$ tells you **how strongly** each flux mode maps into a metabolite-change mode.

* Large $\sigma_k$: there exists a flux combination $\mathbf{v}_k$ that creates a “large” metabolite-change pattern $\mathbf{u}_k$ per unit flux norm.
* Small $\sigma_k$: that direction is **weakly expressed** by the map; numerically it’s close to the nullspace.

Practically, the spectrum ${\sigma_k}$ gives you:

* $\mathrm{rank}(\mathbf{S})$ (how many independent constraints/couplings are in the network),
* a conditioning picture (useful if you’re doing pseudoinverses, least squares flux estimation, etc.),
* a principled **model reduction** knob: keep only the top $K$ modes
  $$
  \mathbf{S}\approx \sum_{k=1}^{K}\sigma_k,\mathbf{u}_k,\mathbf{v}_k^\top
  $$
  to get a low-rank approximation that preserves dominant stoichiometric couplings.

---

## Summary
One concise summry sentence goes here.

> __Key Takeaways:__
>
> Three key takeaways go here.

One concise concluding sentence goes here.
___